### 🔊 Esta aplicação transforma arquivos de **texto** em **áudios narrados (MP3)** usando a API **Gemini TTS** do Google — a mesma tecnologia do app Phonika.

Serve para gerar **notas de áudio**, **notas de esclarecimento ao público**, **comunicados institucionais**, **boletins**, **aulas narradas**, **audiolivros** — ou qualquer narração com instrução personalizada.

*Nota: no modo "Link da pasta do Drive", é necessário dar permissão à aplicação para se conectar ao seu Drive. Apenas você terá acesso ao conteúdo.*

Esta aplicação:
1. Lê os arquivos de texto (`.txt`, `.md` ou **Google Docs**) de uma pasta do seu Google Drive — **ou** recebe arquivos por **upload** do seu computador.
2. Narra cada texto com a voz, o tipo de conteúdo, o tom e o ritmo que você escolher no formulário.
3. Devolve o **MP3 na mesma pasta do Drive** (ou **baixa para o seu computador**, no modo upload).

---

**Antes de começar você precisa de uma chave (gratuita) da API Gemini:**
1. Acesse [aistudio.google.com/apikey](https://aistudio.google.com/apikey) e crie uma chave.
2. No Colab, clique no ícone de **🔑 chave (Secrets)** na barra lateral esquerda, crie um segredo chamado `GEMINI_API_KEY`, cole a chave e ative "Notebook access".
   *(Se preferir, pule esta etapa: o notebook pedirá a chave ao rodar.)*

**Dicas:**
- Textos que já viraram MP3 são **pulados** automaticamente (a menos que você marque "sobrescrever") — pode rodar de novo sem medo.
- Se a geração for interrompida no meio (rede, cota da API), os trechos já gerados ficam em **cache**: basta rodar de novo que ela **retoma de onde parou**.
- A cota gratuita do Gemini TTS é pequena (poucas requisições por dia). Textos longos podem precisar de mais de um dia — a retomada por trecho cuida disso.


In [ ]:
# @title ⚙️ Configuração — preencha o formulário e rode a célula { display-mode: "form" }

# ═════════════════════════════════════════════════════════════════════════════
# Gerar áudios (MP3) a partir de textos com a API Gemini TTS
#
# Portado do app Phonika (Electron): mesma lógica — verificada em produção — de
# fatiamento do texto, instrução de estilo por trecho (voz consistente), limite
# de requisições por minuto, retries com backoff, classificação da cota diária,
# normalização de volume entre trechos e RETOMADA por trecho via cache.
# ═════════════════════════════════════════════════════════════════════════════

# ── 1. Entrada ───────────────────────────────────────────────────────────────
# • "Link da pasta do Drive": lê os textos de uma pasta do seu Google Drive e
#   devolve os MP3 na MESMA pasta.
# • "Upload de arquivos": envia .txt/.md do computador; os MP3 são baixados de
#   volta no final (não usa o Drive e não pede autenticação Google).
MODO_ENTRADA = "Link da pasta do Drive"  # @param ["Link da pasta do Drive", "Upload de arquivos"]

# Link da pasta no Google Drive (usado apenas no modo "Link da pasta do Drive")
PASTA_TEXTOS = ""  # @param {type:"string"}

# ── 2. O que gerar ───────────────────────────────────────────────────────────
# O tipo de conteúdo define a INSTRUÇÃO DE NARRAÇÃO enviada ao Gemini junto com
# cada trecho (a instrução não é falada — o modelo a interpreta como direção).
TIPO_CONTEUDO = "Nota de esclarecimento ao público"  # @param ["Nota de áudio", "Nota de esclarecimento ao público", "Comunicado institucional", "Notícia / boletim informativo", "Aula / explicação didática", "Audiolivro", "Personalizado"]

# Instrução personalizada — usada quando TIPO_CONTEUDO = "Personalizado".
# Descreva livremente como o texto deve ser narrado. Ex.: "Leia como um
# anúncio de utilidade pública, com urgência controlada e dicção muito clara".
INSTRUCAO_PERSONALIZADA = ""  # @param {type:"string"}

TOM = "neutro"  # @param ["neutro", "calmo e acolhedor", "enérgico e vivo", "sério e sóbrio", "alegre e leve", "grave e encorpado", "sussurrado e suave", "personalizado"]

# Tom personalizado — usado quando TOM = "personalizado". Ex.: "com leve tom de
# preocupação, mas transmitindo segurança".
TOM_PERSONALIZADO = ""  # @param {type:"string"}

RITMO = "normal"  # @param ["normal", "mais pausado", "mais rápido"]

# ── 3. Voz e áudio ───────────────────────────────────────────────────────────
VOZ = "Aoede — feminina, leve e expressiva"  # @param ["Aoede — feminina, leve e expressiva", "Kore — feminina, casual e direta", "Leda — feminina, jovem e alegre", "Zephyr — feminina, suave e calma", "Charon — masculina, articulada e neutra", "Puck — masculina, leve e simpática", "Fenrir — masculina, firme e grave", "Orus — masculina, calma e ponderada"]

# Velocidade final do MP3 (aplicada com atempo, preserva o tom da voz)
VELOCIDADE = 1.0  # @param {type:"slider", min:0.7, max:1.3, step:0.05}

# "Leitura pausada": insere um silêncio entre as frases (bom para notas lidas
# em voz oficial e para quem ouve com atenção a cada frase)
LEITURA_PAUSADA = False  # @param {type:"boolean"}
PAUSA_ENTRE_FRASES_MS = 350  # @param {type:"slider", min:100, max:1500, step:50}

# ── 4. Execução ──────────────────────────────────────────────────────────────
# Gerar de novo mesmo que o MP3 já exista na pasta de destino
SOBRESCREVER_MP3_EXISTENTE = False  # @param {type:"boolean"}

# Tamanho máximo de cada trecho enviado à API (caracteres). Menor = mais
# requisições porém mais estável; maior = menos requisições porém mais lento
# por chamada. 3500 é o padrão do Phonika.
TAMANHO_TRECHO = 3500  # @param {type:"slider", min:500, max:8000, step:100}

# Limite de requisições por minuto (Tier gratuito do Gemini TTS = 10 RPM;
# usamos 9 por segurança, como no Phonika)
REQUISICOES_POR_MINUTO = 9  # @param {type:"slider", min:1, max:60, step:1}

MODELO_TTS = "gemini-2.5-flash-preview-tts"  # @param ["gemini-2.5-flash-preview-tts", "gemini-2.5-pro-preview-tts"] {allow-input: true}


# ═════════════════════════════════════════════════════════════════════════════
# A partir daqui é o motor — não precisa mexer.
# ═════════════════════════════════════════════════════════════════════════════

import base64
import hashlib
import io
import os
import re
import subprocess
import time
import wave

import numpy as np
import requests

SAMPLE_RATE = 24000  # PCM 16-bit LE mono 24 kHz — formato fixo do Gemini TTS
NOME_VOZ = VOZ.split("—")[0].strip()

if MODO_ENTRADA == "Link da pasta do Drive" and not PASTA_TEXTOS.strip():
    raise ValueError(
        "❌ Nenhum link da pasta do Google Drive foi fornecido. "
        "Preencha PASTA_TEXTOS ou mude MODO_ENTRADA para 'Upload de arquivos'."
    )
if TIPO_CONTEUDO == "Personalizado" and not INSTRUCAO_PERSONALIZADA.strip():
    raise ValueError(
        "❌ TIPO_CONTEUDO = 'Personalizado' exige preencher INSTRUCAO_PERSONALIZADA."
    )

# ── Chave da API Gemini (Secrets do Colab ou prompt) ─────────────────────────
GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = (userdata.get("GEMINI_API_KEY") or "").strip()
except Exception:
    GEMINI_API_KEY = None
if not GEMINI_API_KEY:
    from getpass import getpass
    GEMINI_API_KEY = getpass(
        "🔑 Cole sua chave da API Gemini (crie grátis em aistudio.google.com/apikey): "
    ).strip()
if not GEMINI_API_KEY:
    raise ValueError("❌ Sem chave da API Gemini — não é possível gerar áudio.")

TTS_URL = (
    f"https://generativelanguage.googleapis.com/v1beta/models/{MODELO_TTS}:generateContent"
)

# ── Autenticação no Google Drive (apenas no modo Drive) ──────────────────────
creds = None
if MODO_ENTRADA == "Link da pasta do Drive":
    from google.colab import drive, auth
    import pickle
    import google.auth
    from google.auth.transport.requests import Request
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

    drive.mount("/content/drive")

    TOKEN_PATH = "/content/drive/MyDrive/token_colab.pickle"
    if os.path.exists(TOKEN_PATH):
        with open(TOKEN_PATH, "rb") as token_file:
            creds = pickle.load(token_file)
    if creds and creds.expired and creds.refresh_token:
        try:
            creds.refresh(Request())
            with open(TOKEN_PATH, "wb") as token_file:
                pickle.dump(creds, token_file)
            print("🔑 Token expirado: renovado com sucesso.")
        except Exception:
            creds = None
    if not creds or not getattr(creds, "valid", False):
        auth.authenticate_user()
        creds, _ = google.auth.default()
        with open(TOKEN_PATH, "wb") as token_file:
            pickle.dump(creds, token_file)
        print(f"🔑 Token salvo em: {TOKEN_PATH}")
    else:
        print(f"🔑 Token reutilizado de: {TOKEN_PATH}")

    # Cache de trechos no próprio Drive → a retomada sobrevive ao reinício da
    # VM do Colab (essencial com a cota diária pequena do TTS gratuito).
    CACHE_DIR = "/content/drive/MyDrive/GerarAudioGemini/cache-trechos"
else:
    CACHE_DIR = "/tmp/gerar-audio-gemini/cache-trechos"
os.makedirs(CACHE_DIR, exist_ok=True)


# ── Instrução de narração (portada/estendida do chunk.ts do Phonika) ─────────
# Mandada IGUAL em todo trecho, ancora o modelo no mesmo tom/voz a cada chamada,
# reduzindo a variação de timbre e entonação entre os pedaços.
TIPOS_CONTEUDO = {
    "Nota de áudio": (
        "Leia o texto a seguir como uma nota de áudio pessoal, em tom natural e "
        "próximo, como quem fala diretamente com o ouvinte"
    ),
    "Nota de esclarecimento ao público": (
        "Leia o texto a seguir como uma nota oficial de esclarecimento ao público, "
        "em tom institucional, sério, claro e respeitoso, com dicção firme e segura"
    ),
    "Comunicado institucional": (
        "Leia o texto a seguir como um comunicado institucional formal, em tom "
        "profissional, objetivo e cordial"
    ),
    "Notícia / boletim informativo": (
        "Leia o texto a seguir como um locutor de rádio profissional apresentando "
        "um boletim informativo, com ritmo dinâmico e dicção muito clara"
    ),
    "Aula / explicação didática": (
        "Leia o texto a seguir como um professor explicando o conteúdo com "
        "clareza, didática e naturalidade, destacando as ideias principais"
    ),
    "Audiolivro": (
        "Leia o texto a seguir como um audiolivro profissional"
    ),
}

TONS = {
    "neutro": "",
    "calmo e acolhedor": "em tom calmo, sereno e acolhedor",
    "enérgico e vivo": "em tom enérgico, com entusiasmo e vivacidade",
    "sério e sóbrio": "em tom sério, formal e sóbrio",
    "alegre e leve": "em tom alegre e leve",
    "grave e encorpado": "com voz mais grave e encorpada",
    "sussurrado e suave": "com voz sussurrada, suave e baixa",
}

RITMOS = {
    "normal": "",
    "mais pausado": "em ritmo mais pausado e cadenciado",
    "mais rápido": "em ritmo um pouco mais ágil, sem atropelar as palavras",
}


def montar_instrucao():
    """Monta o prefixo de direção enviado antes de cada trecho (não é falado)."""
    if TIPO_CONTEUDO == "Personalizado":
        base = INSTRUCAO_PERSONALIZADA.strip().rstrip(".")
    else:
        base = TIPOS_CONTEUDO[TIPO_CONTEUDO]
    extras = []
    tom = TOM_PERSONALIZADO.strip() if TOM == "personalizado" else TONS.get(TOM, "")
    if tom:
        extras.append(tom)
    ritmo = RITMOS.get(RITMO, "")
    if ritmo:
        extras.append(ritmo)
    sufixo = f", {'; '.join(extras)}" if extras else ""
    return (
        f"{base}, com a MESMA voz e entonação constante do começo ao fim{sufixo}. "
        "Não leia esta instrução:"
    )


INSTRUCAO = montar_instrucao()
print(f"\n🎙️ Voz: {VOZ}")
print(f"📣 Instrução de narração: {INSTRUCAO}")


# ── Fatiamento do texto (portado do chunk.ts do Phonika) ─────────────────────
def fatiar_texto(texto, max_len=3500):
    """Fatia o texto em pedaços de até max_len, preferindo fronteiras de
    PARÁGRAFO e, dentro delas, de FRASE — nunca corta no meio de uma frase, a
    menos que ela sozinha estoure max_len."""
    limpo = texto.replace("\r\n", "\n").strip()
    if not limpo:
        return []

    paragrafos = [
        re.sub(r"\s+", " ", p).strip()
        for p in re.split(r"\n[ \t]*\n+", limpo)
        if p.strip()
    ]

    pedacos = []
    atual = ""

    def descarregar():
        nonlocal atual
        if atual.strip():
            pedacos.append(atual.strip())
        atual = ""

    def fatiar_paragrafo(p):
        saida, buf = [], ""
        frases = re.findall(r"[^.!?…]+[.!?…]*\s*", p) or [p]
        for f in frases:
            if len(f) > max_len:
                if buf.strip():
                    saida.append(buf.strip())
                    buf = ""
                for i in range(0, len(f), max_len):
                    saida.append(f[i : i + max_len].strip())
                continue
            if len(buf + f) > max_len:
                if buf.strip():
                    saida.append(buf.strip())
                buf = f
            else:
                buf += f
        if buf.strip():
            saida.append(buf.strip())
        return [s for s in saida if s]

    for p in paragrafos:
        if len(p) > max_len:
            descarregar()
            pedacos.extend(fatiar_paragrafo(p))
            continue
        sep = "\n\n" if atual else ""
        if len(atual + sep + p) > max_len:
            descarregar()
            atual = p
        else:
            atual += sep + p
    descarregar()
    return [c for c in pedacos if c]


def separar_frases(texto):
    """'Leitura pausada': separa cada FRASE por linha em branco — o fatiamento
    reagrupa depois, mantendo as pausas entre frases."""
    limpo = texto.replace("\r\n", "\n").strip()
    if not limpo:
        return limpo
    paras = []
    for para in re.split(r"\n[ \t]*\n+", limpo):
        if not para.strip():
            continue
        frases = re.findall(r"[^.!?…]+[.!?…]*\s*", re.sub(r"\s+", " ", para).strip())
        if not frases:
            paras.append(para.strip())
        else:
            paras.append("\n\n".join(f.strip() for f in frases if f.strip()))
    return "\n\n".join(paras)


# ── Áudio: normalização, WAV, silêncio ───────────────────────────────────────
def normalizar_pcm(pcm):
    """Normaliza o VOLUME para um RMS-alvo constante (~-16 dBFS), senão o volume
    'pula' entre os trechos (cada chamada da API sai mais alto/baixo). Ganho
    limitado e teto de pico para não estourar."""
    n = len(pcm) // 2
    if n == 0:
        return pcm
    amostras = np.frombuffer(pcm[: n * 2], dtype="<i2").astype(np.float64)
    rms = float(np.sqrt(np.mean(amostras**2)))
    if rms < 1:
        return pcm
    ganho = max(0.5, min(3.0, 5000.0 / rms))
    pico = max(1.0, float(np.abs(amostras).max()))
    if pico * ganho > 30000:
        ganho = 30000.0 / pico
    if abs(ganho - 1) < 0.02:
        return pcm
    saida = np.clip(np.round(amostras * ganho), -32768, 32767).astype("<i2")
    return saida.tobytes()


def silencio_pcm(ms):
    """Silêncio (PCM zerado) de `ms` para inserir entre trechos na leitura pausada."""
    return bytes(int(SAMPLE_RATE * ms / 1000) * 2)


def gravar_wav(pcm, caminho):
    with wave.open(caminho, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(SAMPLE_RATE)
        w.writeframes(pcm)


def wav_para_mp3(caminho_wav, caminho_mp3, velocidade=1.0):
    """Transcodifica para MP3 (~165 kbps VBR, ótimo para voz). `atempo` ajusta a
    velocidade preservando o tom, só quando difere de 1.0."""
    cmd = ["ffmpeg", "-y", "-i", caminho_wav, "-codec:a", "libmp3lame", "-q:a", "4"]
    v = max(0.5, min(2.0, velocidade or 1.0))
    if abs(v - 1.0) > 0.001:
        cmd += ["-filter:a", f"atempo={v:.3f}"]
    cmd.append(caminho_mp3)
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)


# ── Chamada à API com RPM, retries e classificação de cota ───────────────────
class ErroGeminiTTS(Exception):
    def __init__(self, mensagem, cota_diaria=False):
        super().__init__(mensagem)
        self.cota_diaria = cota_diaria


MAX_RETRIES = 6
_janela_rpm = []


def _aguardar_slot_rpm():
    """Limitador de REQUISIÇÕES POR MINUTO (janela deslizante de 60s) — global,
    evita o 429 por rajada."""
    limite = max(1, REQUISICOES_POR_MINUTO)
    while True:
        agora = time.time()
        while _janela_rpm and (agora - _janela_rpm[0] > 60 or _janela_rpm[0] > agora):
            _janela_rpm.pop(0)
        if len(_janela_rpm) < limite:
            _janela_rpm.append(agora)
            return
        espera = min(60.25, max(0.25, 60 - (agora - _janela_rpm[0]) + 0.25))
        time.sleep(espera)


def _classificar_429(corpo):
    """Distingue um 429 DIÁRIO (RPD) de um POR-MINUTO (RPM/TPM)."""
    try:
        import json as _json
        s = _json.dumps(corpo.get("error", {}).get("details", []))
    except Exception:
        s = ""
    diario = bool(re.search(r"PerDay|per day|RequestsPerDay", s, re.I))
    m = re.search(r'"retryDelay"\s*:\s*"(\d+(?:\.\d+)?)s"', s)
    atraso = min(65.0, float(m.group(1)) + 0.5) if m else 0.0
    return diario, atraso


def buscar_pcm(texto, tentativa=0):
    """Chama a API e devolve o PCM do trecho. Re-tenta (backoff) em TODOS os
    erros transitórios — 429 por-minuto, 5xx, timeout/rede e 200 sem áudio. Só
    4xx 'de verdade' (chave inválida) e o 429 DIÁRIO são fatais."""
    _aguardar_slot_rpm()
    corpo = {
        "contents": [{"parts": [{"text": texto}]}],
        "generationConfig": {
            "responseModalities": ["AUDIO"],
            "speechConfig": {
                "voiceConfig": {"prebuiltVoiceConfig": {"voiceName": NOME_VOZ}}
            },
        },
    }
    # Timeout ESCALA com o tamanho do trecho: trecho maior gera mais áudio e
    # leva mais tempo. Base 120s, até 300s.
    timeout_s = min(300, max(120, int(len(texto) * 0.07)))
    atraso_backoff = min(30, 2 * (2**tentativa))
    try:
        resp = requests.post(
            f"{TTS_URL}?key={GEMINI_API_KEY}",
            json=corpo,
            timeout=timeout_s,
        )
    except requests.RequestException as e:
        if tentativa < MAX_RETRIES:
            time.sleep(atraso_backoff)
            return buscar_pcm(texto, tentativa + 1)
        raise ErroGeminiTTS(f"Gemini TTS: erro de rede/timeout — {e}")

    if 500 <= resp.status_code < 600 and tentativa < MAX_RETRIES:
        time.sleep(atraso_backoff)
        return buscar_pcm(texto, tentativa + 1)

    if resp.status_code == 429:
        try:
            j = resp.json()
        except Exception:
            j = {}
        diario, atraso_api = _classificar_429(j)
        if diario:
            raise ErroGeminiTTS(
                "cota DIÁRIA da API esgotada — reseta à meia-noite no Pacífico. "
                "Os trechos já gerados ficaram em cache: rode de novo amanhã que "
                "a geração RETOMA de onde parou.",
                cota_diaria=True,
            )
        if tentativa < MAX_RETRIES:
            ra = resp.headers.get("retry-after", "")
            atraso = (
                atraso_api
                if atraso_api > 0
                else (min(65.0, float(ra)) if ra.replace(".", "", 1).isdigit() else 0)
            ) or min(60, 12 * (2**tentativa))
            time.sleep(atraso)
            return buscar_pcm(texto, tentativa + 1)

    if not resp.ok:
        try:
            msg = resp.json().get("error", {}).get("message", f"HTTP {resp.status_code}")
        except Exception:
            msg = f"HTTP {resp.status_code}"
        raise ErroGeminiTTS(f"Gemini TTS: {msg}")

    try:
        dados = (
            resp.json()["candidates"][0]["content"]["parts"][0]["inlineData"]["data"]
        )
    except Exception:
        dados = None
    if not dados:
        if tentativa < MAX_RETRIES:
            time.sleep(atraso_backoff)
            return buscar_pcm(texto, tentativa + 1)
        raise ErroGeminiTTS("Gemini TTS: resposta sem áudio")
    return base64.b64decode(dados + "=" * (-len(dados) % 4))


# ── Cache de TRECHOS (retomada) ──────────────────────────────────────────────
# Cada trecho gerado (PCM já normalizado) é gravado em disco com chave =
# hash(modelo+voz+instrução+texto). Se a geração falhar no meio (ex.: cota
# esgotou no trecho 26/98), ao rodar de novo os trechos prontos são lidos do
# cache e a geração RETOMA do ponto que parou. Ao concluir um arquivo inteiro,
# o cache dos seus trechos é apagado.
def _chave_trecho(texto):
    base = f"{MODELO_TTS}\n{NOME_VOZ}\n{texto}"
    return hashlib.sha1(base.encode("utf-8")).hexdigest()


def _limpar_cache_antigo(dias=7):
    corte = time.time() - dias * 24 * 3600
    try:
        for nome in os.listdir(CACHE_DIR):
            if not nome.endswith(".pcm"):
                continue
            f = os.path.join(CACHE_DIR, nome)
            try:
                if os.path.getmtime(f) < corte:
                    os.remove(f)
            except OSError:
                pass
    except OSError:
        pass


def sintetizar_texto(texto, rotulo=""):
    """Sintetiza o texto INTEIRO: fatia, gera trecho a trecho (com cache e
    retomada), normaliza o volume e concatena o PCM (com silêncio entre trechos
    na leitura pausada). Devolve o PCM completo."""
    preparado = separar_frases(texto) if LEITURA_PAUSADA else texto
    pedacos = fatiar_texto(preparado, TAMANHO_TRECHO)
    if not pedacos:
        raise ErroGeminiTTS("texto vazio")
    com_instrucao = [f"{INSTRUCAO}\n\n{p}" for p in pedacos]

    gap = silencio_pcm(PAUSA_ENTRE_FRASES_MS) if LEITURA_PAUSADA else b""
    partes, arquivos_cache = [], []
    em_cache = 0
    for i, trecho in enumerate(com_instrucao):
        arq = os.path.join(CACHE_DIR, f"{_chave_trecho(trecho)}.pcm")
        arquivos_cache.append(arq)
        if os.path.exists(arq):
            with open(arq, "rb") as fh:
                pcm = fh.read()
            em_cache += 1
        else:
            print(f"   🎙️ {rotulo}trecho {i + 1}/{len(com_instrucao)}...")
            pcm = normalizar_pcm(buscar_pcm(trecho))
            try:
                with open(arq, "wb") as fh:
                    fh.write(pcm)
            except OSError:
                pass  # cache é best-effort; segue mesmo sem gravar
        if gap and i > 0:
            partes.append(gap)
        partes.append(pcm)
    if em_cache:
        print(f"   ♻️  {em_cache} trecho(s) reaproveitado(s) do cache (retomada).")

    # Arquivo concluído inteiro → não precisa mais dos trechos em cache.
    for f in arquivos_cache:
        try:
            os.remove(f)
        except OSError:
            pass
    return b"".join(partes)


def gerar_mp3(texto, caminho_mp3, rotulo=""):
    pcm = sintetizar_texto(texto, rotulo)
    caminho_wav = "/tmp/_gerar_audio_tmp.wav"
    gravar_wav(pcm, caminho_wav)
    try:
        wav_para_mp3(caminho_wav, caminho_mp3, VELOCIDADE)
    finally:
        try:
            os.remove(caminho_wav)
        except OSError:
            pass
    duracao = len(pcm) / 2 / SAMPLE_RATE / max(0.5, min(2.0, VELOCIDADE))
    print(f"   ✅ MP3 gerado ({duracao / 60:.1f} min): {os.path.basename(caminho_mp3)}")


# ── Modo "Link da pasta do Drive" ────────────────────────────────────────────
EXTENSOES_TEXTO = (".txt", ".md", ".markdown", ".text")


def _nome_saida(nome):
    base = nome
    for ext in EXTENSOES_TEXTO:
        if base.lower().endswith(ext):
            base = base[: -len(ext)]
            break
    return base + ".mp3"


def gerar_da_pasta_do_drive():
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

    folder_id = PASTA_TEXTOS.split("folders/")[-1].split("?")[0].strip("/")
    print(f"\n📁 ID da pasta: {folder_id}")
    service = build("drive", "v3", cache_discovery=False, credentials=creds)

    # Listar todos os arquivos da pasta
    itens, page_token = [], None
    while True:
        resposta = (
            service.files()
            .list(
                q=(
                    f"'{folder_id}' in parents and trashed = false "
                    "and mimeType != 'application/vnd.google-apps.shortcut'"
                ),
                fields="nextPageToken, files(id, name, mimeType)",
                pageSize=1000,
                pageToken=page_token,
            )
            .execute()
        )
        itens.extend(resposta.get("files", []))
        page_token = resposta.get("nextPageToken")
        if not page_token:
            break

    def eh_texto(item):
        if item["mimeType"] == "application/vnd.google-apps.document":
            return True
        if item["mimeType"].startswith("text/"):
            return True
        return item["name"].lower().endswith(EXTENSOES_TEXTO)

    textos = sorted((i for i in itens if eh_texto(i)), key=lambda i: i["name"])
    # MP3 já existentes na pasta → id por nome, para pular/sobrescrever
    mp3_existentes = {
        i["name"]: i["id"]
        for i in itens
        if i["name"].lower().endswith(".mp3") or i["mimeType"] == "audio/mpeg"
    }

    if not textos:
        print("⚠️ Nenhum arquivo de texto (.txt, .md ou Google Docs) encontrado na pasta.")
        return

    print(f"📋 Arquivos de texto encontrados: {len(textos)}")
    resumo = {"ok": 0, "pulados": 0, "erros": 0}

    for idx, item in enumerate(textos, 1):
        nome_mp3 = _nome_saida(item["name"])
        print(f"\n{'=' * 60}")
        print(f"🔊 [{idx}/{len(textos)}] {item['name']}  →  {nome_mp3}")
        print(f"{'=' * 60}")

        if nome_mp3 in mp3_existentes and not SOBRESCREVER_MP3_EXISTENTE:
            print("   ⏭️  Já existe na pasta — pulado (marque 'sobrescrever' para regenerar).")
            resumo["pulados"] += 1
            continue

        # Baixar o texto (Google Docs é exportado como texto puro)
        try:
            if item["mimeType"] == "application/vnd.google-apps.document":
                req = service.files().export_media(
                    fileId=item["id"], mimeType="text/plain"
                )
            else:
                req = service.files().get_media(fileId=item["id"])
            buf = io.BytesIO()
            downloader = MediaIoBaseDownload(buf, req)
            concluido = False
            while not concluido:
                _, concluido = downloader.next_chunk()
            bruto = buf.getvalue()
            try:
                texto = bruto.decode("utf-8")
            except UnicodeDecodeError:
                texto = bruto.decode("latin-1")
        except Exception as e:
            print(f"   ❌ Erro ao baixar o texto: {e}")
            resumo["erros"] += 1
            continue

        if not texto.strip():
            print("   ⚠️ Arquivo vazio — pulado.")
            resumo["pulados"] += 1
            continue

        # Gerar e enviar de volta à MESMA pasta
        caminho_local = f"/tmp/{nome_mp3.replace('/', '_')}"
        try:
            gerar_mp3(texto, caminho_local)
        except ErroGeminiTTS as e:
            print(f"   ❌ {e}")
            resumo["erros"] += 1
            if e.cota_diaria:
                print("\n🛑 Parando: a cota diária da API esgotou. Rode de novo amanhã — retoma de onde parou.")
                break
            continue

        try:
            media = MediaFileUpload(caminho_local, mimetype="audio/mpeg")
            if nome_mp3 in mp3_existentes:
                service.files().update(
                    fileId=mp3_existentes[nome_mp3], media_body=media
                ).execute()
                print("   ☁️  MP3 atualizado na pasta do Drive (sobrescrito).")
            else:
                service.files().create(
                    body={"name": nome_mp3, "parents": [folder_id]},
                    media_body=media,
                    fields="id",
                ).execute()
                print("   ☁️  MP3 enviado para a pasta do Drive.")
            resumo["ok"] += 1
        except Exception as e:
            print(f"   ❌ Erro ao enviar o MP3 ao Drive: {e}")
            resumo["erros"] += 1
        finally:
            try:
                os.remove(caminho_local)
            except OSError:
                pass

    print(f"\n{'=' * 60}")
    print("🎉 Processamento concluído!")
    print(
        f"   Gerados: {resumo['ok']} | Pulados: {resumo['pulados']} | Erros: {resumo['erros']}"
    )
    print(f"   📁 Pasta: https://drive.google.com/drive/folders/{folder_id}")
    print(f"{'=' * 60}")


# ── Modo "Upload de arquivos" ────────────────────────────────────────────────
def gerar_de_uploads():
    from google.colab import files

    print("📤 Selecione o(s) arquivo(s) de texto (.txt ou .md) do seu computador...")
    enviados = files.upload()
    if not enviados:
        print("⚠️ Nenhum arquivo enviado. Encerrando.")
        return

    resumo = {"ok": 0, "erros": 0}
    para_baixar = []
    nomes = sorted(enviados.keys())
    for idx, nome in enumerate(nomes, 1):
        conteudo = enviados[nome]
        print(f"\n{'=' * 60}")
        print(f"🔊 [{idx}/{len(nomes)}] {nome}")
        print(f"{'=' * 60}")
        if not nome.lower().endswith(EXTENSOES_TEXTO):
            print("   ⚠️ Ignorado (não é .txt/.md).")
            continue
        try:
            texto = conteudo.decode("utf-8")
        except UnicodeDecodeError:
            texto = conteudo.decode("latin-1")
        if not texto.strip():
            print("   ⚠️ Arquivo vazio — pulado.")
            continue
        caminho_mp3 = f"/tmp/{_nome_saida(nome).replace('/', '_')}"
        try:
            gerar_mp3(texto, caminho_mp3)
            para_baixar.append(caminho_mp3)
            resumo["ok"] += 1
        except ErroGeminiTTS as e:
            print(f"   ❌ {e}")
            resumo["erros"] += 1
            if e.cota_diaria:
                print("\n🛑 Parando: a cota diária da API esgotou.")
                break

    print(f"\n{'=' * 60}")
    print(f"🎉 Processamento concluído! Gerados: {resumo['ok']} | Erros: {resumo['erros']}")
    if para_baixar:
        print("   💾 Baixando os MP3 para o seu computador...")
    print(f"{'=' * 60}")
    for caminho in para_baixar:
        files.download(caminho)


# ═════════════════════════════════════════════════════════════════════════════
# EXECUÇÃO
# ═════════════════════════════════════════════════════════════════════════════
_limpar_cache_antigo()
if MODO_ENTRADA == "Link da pasta do Drive":
    gerar_da_pasta_do_drive()
else:
    gerar_de_uploads()
